# PhysioCoach-NLP: retrain Flan-T5 on the paraphrased dataset (Kaggle)

Needs, in the right-hand panel:
- **Input:** your dataset containing `physiocoach-nlp.zip` and `data_para.zip`
- **Settings -> Accelerator:** GPU T4 x2 (or GPU P100)
- **Settings -> Internet:** On

Run with **Save Version -> Save & Run All (Commit)**. It keeps running even if you close the browser.
When it finishes, download `flan-t5-coach-para.zip` from the version's **Output** tab.

## 1. Check the GPU

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 2. Find and unpack your files

In [ ]:
import glob, os, shutil, zipfile

os.environ["CUDA_VISIBLE_DEVICES"] = "0"   # use one GPU: simpler and enough for flan-t5-small
INP, WORK, OUT = "/kaggle/input", "/tmp/work", "/kaggle/working"
shutil.rmtree(WORK, ignore_errors=True)
os.makedirs(WORK)
print("Files found in your input:")
for p in sorted(glob.glob(f"{INP}/**/*", recursive=True))[:40]:
    print("  ", p)

# Kaggle may keep uploaded zips as zips or unpack them into folders: handle both.
project_zip = glob.glob(f"{INP}/**/physiocoach-nlp.zip", recursive=True)
if project_zip:
    zipfile.ZipFile(project_zip[0]).extractall(WORK)
else:
    found = glob.glob(f"{INP}/**/pyproject.toml", recursive=True)
    assert found, "Project not found. Add your dataset with 'Add Input' (right panel)."
    shutil.copytree(os.path.dirname(found[0]), f"{WORK}/physiocoach-nlp")
PROJ = f"{WORK}/physiocoach-nlp"
assert os.path.exists(f"{PROJ}/pyproject.toml"), "physiocoach-nlp folder looks wrong"

data_zip = glob.glob(f"{INP}/**/data_para.zip", recursive=True)
if data_zip:
    zipfile.ZipFile(data_zip[0]).extractall(f"{WORK}/_dp")
    train_files = glob.glob(f"{WORK}/_dp/**/train.jsonl", recursive=True)
else:
    train_files = glob.glob(f"{INP}/**/train.jsonl", recursive=True)
assert train_files, "data_para not found. Upload data_para.zip to the same dataset."
shutil.copytree(os.path.dirname(train_files[0]), f"{PROJ}/data_para")

os.chdir(PROJ)
for name in ["train.jsonl", "val.jsonl", "test.jsonl"]:
    n = sum(1 for _ in open(f"data_para/{name}", encoding="utf-8"))
    print(f"{name}: {n} pairs")

## 3. Install

In [ ]:
os.chdir(PROJ)
!pip install -q -e ".[ml]"

## 4. Train (about 20-40 minutes)

In [ ]:
os.chdir(PROJ)
!python scripts/train_t5.py --data data_para --model google/flan-t5-small --out models/flan-t5-coach-para --epochs 6 2>&1 | tee /kaggle/working/train_log_para.txt

## 5. Quick look at the new model

In [ ]:
os.chdir(PROJ)
!PHYSIOCOACH_T5_DIR=models/flan-t5-coach-para PHYSIOCOACH_TRANSLATOR=none PHYSIOCOACH_TTS=none physiocoach demo -n 8 --backend t5 2>&1 | tee /kaggle/working/quick_look.txt

## 6. Package the model for download

In [ ]:
import subprocess
os.chdir(PROJ)
subprocess.run(["zip", "-qr", f"{OUT}/flan-t5-coach-para.zip", "models/flan-t5-coach-para",
                "-x", "models/flan-t5-coach-para/checkpoints/*"], check=True)
shutil.copy("models/flan-t5-coach-para/val_faithfulness.json", f"{OUT}/val_faithfulness_para.json")
print("Ready to download from the Output tab:")
for f in sorted(os.listdir(OUT)):
    print(f"  {f}  ({os.path.getsize(os.path.join(OUT, f)) / 1e6:.1f} MB)")